In [ ]:
import os
import numpy as np
import pandas as pd
from skimage import io, transform
from os import rename, listdir
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
import lightgbm as lgb
import xgboost as xgb
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import Augmentor

In [ ]:
img='./data/images'
train_csv_file = "trainset.csv"
test_csv_file = "testset.csv"

# augmentation 
**********************************************************************

In [ ]:
# img='./data/MalignantImages/'
# p = Augmentor.Pipeline(img)
# p.rotate(probability=0.7, max_left_rotation=10, max_right_rotation=10)
# p.zoom(probability=0.5, min_factor=1.1, max_factor=1.5)
# p.sample(152)


In [ ]:
# from os import rename, listdir
# folder = "./data/MalignantImages/output/"
# for root, dirs, files in os.walk(folder):
#      for i,f in enumerate(files):
#         absname = os.path.join(root, f)
#         newname = os.path.join(root, (str(i)+'.png'))
#         os.rename(absname, newname)

In [ ]:
# new = pd.DataFrame({'Id': np.arange(152),'Class': 'Malignant'})
# # new.to_csv('test.csv', index=False)
# new

In [ ]:
# df_new=pd.concat([df,new ], sort=False)

In [ ]:
# img='./data/NormalImages'
# import Augmentor
# p = Augmentor.Pipeline(img)
# p.shear(probability=0.5,max_shear_left=10,max_shear_right=10)
# p.sample(209)

In [ ]:
# folder = "./data/NormalImages/output/"
# for root, dirs, files in os.walk(folder):
#      for i,f in enumerate(files):
#         i=i+200
#         absname = os.path.join(root, f)
#         newname = os.path.join(root, (str(i)+'.png'))
#         os.rename(absname, newname)

In [ ]:
# new1 = pd.DataFrame({'Id': np.arange(200,409),'Class': 'Normal'})

In [ ]:
# df_new1=pd.concat([df_new,new1 ], sort=False)

In [ ]:
# df_new1=df_new1.sample(frac=1)
# df_new1=df_new1.drop('Unnamed: 0',axis=1)

In [ ]:
# df_new1.to_csv('df_new1.csv')

****************************************************************

In [ ]:
img='./data/img/'
train_csv_file = "trainset.csv"
test_csv_file = "testset.csv"

In [ ]:
def load_preprocess_image(image_path, target_size):
    img = io.imread(image_path)
    img = transform.resize(img, target_size)
    img = img.flatten()  # Convert image to 1D array
    return img

In [ ]:
def load_dataset_from_csv(csv_path, data_dir, target_size):
    df = pd.read_csv(csv_path)  
    global X,y
    X=[]
    y=[]
    for index, row in df.iterrows():
        image_name = str(row['Id']) + ".png"  
        image_path = os.path.join(data_dir, image_name)
        y_value = row['Class']
        img = load_preprocess_image(image_path, target_size)
        X.append(img)
        y.append(y_value)

    return np.array(X), np.array(y)

In [ ]:
def load_dataset_from1_csv(csv_path, data_dir, target_size):
    df = pd.read_csv(csv_path)  
    X_test=[]
    for index, row in df.iterrows():
        image_name = str(row['Id']) + ".png"  
        image_path = os.path.join(data_dir, image_name)
        img = load_preprocess_image(image_path, target_size)
        X_test.append(img)
    return np.array(X_test)

In [ ]:
target_size = (100, 100)
b='./data/images/'
test_one = load_dataset_from1_csv(test_csv_file, b, target_size)

In [ ]:
# without augmentation
target_size = (100, 100)
img='./data/img/'
X_train, y_train = load_dataset_from_csv(train_csv_file, img, target_size)


In [ ]:
# with augmentation
df_augmentation='df_new1.csv'
img='./data/img/'
X_train_augm, y_train_augm = load_dataset_from_csv(df_augmentation, img, target_size)

In [ ]:
pca = PCA(n_components=100)
pca.fit(X_train)
X_train_pca=pca.transform(X_train)

# pca.explained_variance_ratio_.sum()

In [ ]:
X_train_augm_pca=pca.transform(X_train_augm)

In [ ]:

X_train1, X_test1, y_train1, y_test1 = train_test_split(X_train, y_train, test_size=0.25, random_state=42,stratify=y_train)
X_train_aug, X_test_aug, y_train_aug, y_test_aug = train_test_split(X_train_augm, y_train_augm, test_size=0.25, random_state=42,stratify=y_train_augm)

#  100X100

In [ ]:
# without augmentation

model = RandomForestClassifier(random_state=15,n_estimators=300,max_depth=10)
model.fit(X_train1, y_train1)
y_pred = model.predict(X_test1)
print(classification_report(y_test1, y_pred))

In [ ]:
# with augmentation
model = RandomForestClassifier(random_state=15,n_estimators=300,max_depth=10)
model.fit(X_train_aug, y_train_aug)
y_pred = model.predict(X_test_aug)
print(classification_report(y_test_aug, y_pred))

In [ ]:
# without augmentation
knn = KNeighborsClassifier(n_neighbors=2)
knn.fit(X_train1, y_train1)
y_pred = knn.predict(X_test1)
print(classification_report(y_test1, y_pred))

In [ ]:
# with augmentation
knn = KNeighborsClassifier(n_neighbors=2)
knn.fit(X_train_aug, y_train_aug)
y_pred = knn.predict(X_test_aug)
print(classification_report(y_test_aug, y_pred))

In [ ]:
# without augmentation
clf = MultinomialNB()
clf.fit(X_train1, y_train1)
y_pred = clf.predict(X_test1)
print(classification_report(y_test1, y_pred))


In [ ]:
# with augmentation
clf = MultinomialNB()
clf.fit(X_train_aug, y_train_aug)
y_pred = clf.predict(X_test_aug)
print(classification_report(y_test_aug, y_pred))

In [ ]:
# without augmentation
svm=SVC()
svm.fit(X_train1, y_train1)
y_pred = svm.predict(X_test1)
print(classification_report(y_test1, y_pred))


In [ ]:
# with augmentation
svm=SVC()
svm.fit(X_train_aug, y_train_aug)
y_pred = svm.predict(X_test_aug)
print(classification_report(y_test_aug, y_pred))

In [ ]:
def encoding(data):
    for i in range(len(data)):
        if data[i] == 'Malignant':
            data[i] = 0
 
        if data[i] == 'Normal':
            data[i] = 1

        if data[i] == 'Benign':
            data[i] = 2
    integer_values = [int(value) for value in data]  
    return integer_values



In [ ]:
integer_values=encoding(y_train)
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_train_pca, integer_values, test_size=0.25, random_state=42,stratify=y_train)

In [ ]:
integer_values_aug=encoding(y_train_augm)
X_train_aug_p, X_test_aug_p, y_train_aug_p, y_test_aug_p = train_test_split(X_train_augm_pca, integer_values_aug, test_size=0.25, random_state=42,stratify=y_train_augm)


In [ ]:
# without augmentation and PCA
import xgboost as xgb
clf=xgb.XGBClassifier()
clf.fit(X_train_p, y_train_p)
y_pred1 = clf.predict(X_test_p)
print(classification_report(y_test_p, y_pred1))

In [ ]:
# with augmentation and PCA
clf=xgb.XGBClassifier()
clf.fit(X_train_aug_p, y_train_aug_p)
y_pred1 = clf.predict(X_test_aug_p)
print(classification_report(y_test_aug_p, y_pred1))

In [ ]:
# without augmentation and PCA
model = lgb.LGBMClassifier()
model.fit(X_train_p, y_train_p)
y_pred = model.predict(X_test_p)

In [ ]:
print(classification_report(y_test_p, y_pred))

In [ ]:
# with augmentation and PCA
model = lgb.LGBMClassifier()
model.fit(X_train_aug_p, y_train_aug_p)
y_pred = model.predict(X_test_aug_p)

In [ ]:
print(classification_report(y_test_aug_p, y_pred))

In [ ]:
#Learning Curve:
#x, y
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve
model =RandomForestClassifier()#With the best parameters

train_sizes, train_scores, test_scores = learning_curve(model, X_train_aug_p, y_train_aug_p, cv=5,
                                                        verbose = 1, train_sizes=np.linspace(0.01, 1.0, 50))

#Calculating the mean and standard deviation of the train and test scores
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)

test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)


#Plotting lines and band of the learning curve
plt.subplots(1, figsize=(10,10))
plt.plot(train_sizes, train_mean, '--', color="#111111",  label="Training score")
plt.plot(train_sizes, test_mean, color="#111111", label="Cross-validation score")

plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, color="#DDDDDD")
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, color="#DDDDDD")

plt.title("Learning Curve")
plt.xlabel("Training Set Size"), plt.ylabel("f1 Score"), plt.legend(loc="best")
plt.tight_layout()
plt.show()